In [ ]:
import sys
import os

# Si el notebook se abre desde notebooks/, sube un nivel a CPMP-Framework/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print('CWD:', os.getcwd())
print('src:', src_path)

In [ ]:
# Compilar el solver FRG (requiere g++ — correr desde WSL si estás en Windows).
# Si el binario ya existe y funciona, puedes saltarte esta celda.
!g++ Codigo_C_solver/Greedy.cpp Codigo_C_solver/Layout.cpp Codigo_C_solver/Bsg.cpp Codigo_C_solver/main_cpmp.cpp -o Codigo_C_solver/frg -O3 -std=c++11
!chmod +x Codigo_C_solver/frg
print('Solver listo:', os.path.abspath('Codigo_C_solver/frg'))

In [ ]:
from generation.instances import generate_instances
from generation.data import generate_data
from generation.adapters import TacticalStackMatrixAdapter, DefaultMovesAdapter
from settings import DATA_FOLDER

# Mismas configuraciones que V9 victorioso
configuraciones = [
    # (H, S, N, amount)
    (5, 4, 15, 50000),   # Tableros pequeños (fáciles)
    (7, 5, 25, 100000),  # Benchmark actual (intermedios)
    (10, 6, 45, 50000),  # Tableros grandes y altos (difíciles)
    (6, 7, 30, 50000)    # Tableros anchos pero bajos
]

r_desorden = 50
max_steps = 50
seed = 42

print('Iniciando pipeline de generación V10 (TacticalStackMatrixAdapter)...\n')

for H, S, N, amount in configuraciones:
    filename = f'E{S}-{N}-H{H}_V10'
    print(f'--- Configuración: {S} Stacks, {N} Contenedores, Altura {H} ---')

    print(f'Generando {amount} instancias brutas...')
    generate_instances(filename, H, S, N, amount, r_desorden, seed)

    print('Resolviendo y empaquetando datos tácticos (5D)...')
    layout_adapter = TacticalStackMatrixAdapter()
    moves_adapter = DefaultMovesAdapter()

    generate_data(filename, H, max_steps, layout_adapter, moves_adapter)
    print(f'Finalizado: {DATA_FOLDER / (filename + ".data")}\n')

print('Generacion V10 terminada!')